# Speexx Panel — Colab / Kaggle

Cloud copy of the panel. Logic is identical to the local project; only the
environment layer differs (headless Chromium, no-sandbox, Linux deps, pyngrok
tunnel). See `CLOUD.md` in this folder.

Cells, in order:
1. **Setup** — venv, deps, Chromium, Linux libs, pyngrok
2. **Secrets** — `NGROK_AUTHTOKEN`, optional first admin password
3. **Data dir** — persistent on Kaggle; Drive-optional on Colab
4. **Launch** — panel + public tunnel URL
5. **Monitor / stop**

Never store credentials or the authtoken in the notebook. Never print secrets.

In [ ]:
# 1) Setup — run once per fresh session (~2-4 min).
#    Kaggle: if the folder arrived as a dataset it is under /kaggle/input
#    (read-only), so it is copied into /kaggle/working first.
import os, glob, shutil, subprocess

proj = os.getcwd()
if not os.path.exists("cloud_setup.sh"):
    hits = glob.glob("/kaggle/input/**/cloud_setup.sh", recursive=True)
    if hits:
        src = os.path.dirname(hits[0])
        dst = "/kaggle/working/" + os.path.basename(src)
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
        proj = dst
    else:
        raise SystemExit("cloud_setup.sh not found — cd to the project folder first")
os.chdir(proj)
print("project dir:", proj)
r = subprocess.run(["bash", "cloud_setup.sh"])
assert r.returncode == 0, "setup failed — see the output above"


In [ ]:
# 2) Secrets — Kaggle Secrets / Colab userdata, or fill the fallbacks below.
#    Secret names: NGROK_AUTHTOKEN, SPEEXX_PANEL_ADMIN_PASSWORD (optional).
import os

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return None

token = _secret("NGROK_AUTHTOKEN")
admin_pw = _secret("SPEEXX_PANEL_ADMIN_PASSWORD")
# fallbacks (edit here if you do not use a secrets store — then DO NOT SAVE the notebook):
token = token or ""
admin_pw = admin_pw or ""

if token:
    os.environ["NGROK_AUTHTOKEN"] = token
    print("NGROK_AUTHTOKEN: set")
else:
    print("NGROK_AUTHTOKEN: NOT set — the panel will run without a public URL")
if admin_pw:
    os.environ["SPEEXX_PANEL_ADMIN_PASSWORD"] = admin_pw
    os.environ.setdefault("SPEEXX_PANEL_ADMIN_USERNAME", "admin")
    print("first admin will be bootstrapped")
else:
    print("no admin password set — the first person to sign up owns the panel")

In [ ]:
# 3) Data dir — where panel.db, the vault key and job logs live.
#    Kaggle: /kaggle/working survives the session. Colab: /content does not —
#    uncomment the Drive block to keep saved logins across sessions.
import os

if "/kaggle" in os.getcwd() or os.path.exists("/kaggle/working"):
    os.environ["SPEEXX_PANEL_DATA_DIR"] = "/kaggle/working/panel-data"
else:
    os.environ.setdefault("SPEEXX_PANEL_DATA_DIR", "/content/panel-data")
    # from google.colab import drive
    # drive.mount('/content/drive')
    # os.environ["SPEEXX_PANEL_DATA_DIR"] = "/content/drive/MyDrive/panel-data"
print("data dir:", os.environ["SPEEXX_PANEL_DATA_DIR"])

In [ ]:
# 4) Launch — panel on 127.0.0.1:8787 plus the ngrok tunnel.
#    Prints the public URL; first visit in a new browser shows ngrok's
#    'Visit Site' warning page once. Background so the cell returns.
!rm -f data/tunnel-url.txt /tmp/speexx-tunnel.log
!nohup ./run.sh 8787 --with-tunnel > /tmp/speexx-panel.log 2>&1 &
import time, pathlib
for _ in range(90):
    f = pathlib.Path("data/tunnel-url.txt")
    if f.exists() and f.read_text().strip():
        break
    time.sleep(2)
url = f.read_text().strip() if f.exists() else ""
print()
print("=== PANEL PUBLIC URL ===")
print(url if url else "(tunnel URL did not appear — check /tmp/speexx-tunnel.log)")
print("=========================")
print("local: http://127.0.0.1:8787")

In [ ]:
# 5a) Monitor — panel log and the latest runner log
!tail -n 30 /tmp/speexx-panel.log 2>/dev/null
!ls -t $SPEEXX_PANEL_DATA_DIR/logs 2>/dev/null | head -5

In [ ]:
# 5b) Stop the panel and its runners (conservative patterns, same as stop.sh)
!./stop.sh